# Cenário 14: Correlação entre a Fortuna de Musk e a População de Rua nos EUA

## Contexto
O **HUD (Department of Housing and Urban Development)** realiza anualmente o *Point-in-Time Count* (PIT), um censo noturno da população em situação de rua nos Estados Unidos. Entre 2010 e 2016, os EUA conseguiram **reduzir** o número de sem-teto graças a políticas de *Housing First*. A partir de 2019, e especialmente pós-2020, o número voltou a crescer — atingindo o recorde histórico de **771.480 pessoas** em 2024.

Ao mesmo tempo, a fortuna de Elon Musk cresceu de US$ 2 bilhões (2012) para US$ 1 trilhão (2026), a maior acumulação de riqueza individual da história.

> ⚠️ **Nota metodológica:** O PIT Count de 2021 foi excluído da análise, pois a pandemia de COVID-19 afetou severamente a metodologia de coleta, tornando os dados incomparáveis com outros anos. A correlação estatística **não implica causalidade direta** — ambos os fenômenos são reflexos de políticas habitacionais, mercado imobiliário, desigualdade sistêmica e ciclos econômicos.

**Fontes:**
- [HUD – 2024 Annual Homelessness Assessment Report (AHAR)](https://www.huduser.gov/portal/publications/2024-ahar-part-1-pit-estimates-of-homelessness.html)
- [National Alliance to End Homelessness – HUD 2024 Report](https://endhomelessness.org/media/news-releases/hud-releases-2024-annual-homelessness-assessment-report/)
- [Bloomberg Billionaires Index](https://www.bloomberg.com/billionaires/)
- [Forbes Real-Time Billionaires](https://www.forbes.com/real-time-billionaires/)
- [Bipartisan Policy Center – Homelessness at a Record High](https://bipartisanpolicy.org/article/homelessness-at-a-record-high-key-takeaways-from-the-2024-pit-count/)


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

MUSK_USD = 1_000_000_000_000
USD_BRL  = 5.71
DATA     = 'data/'

# Carregar dados
rua_eua  = pd.read_csv(DATA + 'historico_pop_rua_eua.csv')
musk_h   = pd.read_csv(DATA + 'historico_fortuna_musk.csv')

# Último registro de Musk por ano
musk_ano = musk_h.groupby('ano')['fortuna_bilhoes_usd'].last().reset_index()

# Merge por ano
df = pd.merge(rua_eua[['ano', 'populacao_rua']], musk_ano, on='ano', how='inner')
df = df.sort_values('ano').reset_index(drop=True)

print("Dados combinados (EUA):")
print(df.to_string(index=False))

# Correlações
r_pearson  = np.corrcoef(df['fortuna_bilhoes_usd'], df['populacao_rua'])[0, 1]
# Spearman (ranques)
rank_m = df['fortuna_bilhoes_usd'].rank()
rank_h = df['populacao_rua'].rank()
r_spearman = np.corrcoef(rank_m, rank_h)[0, 1]

print(f"\nCorrelação de Pearson:  {r_pearson:.4f}")
print(f"Correlação de Spearman: {r_spearman:.4f}")


Dados combinados (EUA):
 ano  populacao_rua  fortuna_bilhoes_usd
2012         633782                  2.0
2013         610042                  4.0
2014         578424                  9.0
2015         564708                 13.0
2016         549928                 11.0
2017         553742                 14.0
2018         552830                 23.0
2019         567715                 20.0
2020         580466                165.0
2022         582462                137.0
2023         653104                232.0
2024         771480                432.0

Correlação de Pearson:  0.8401
Correlação de Spearman: 0.2727


In [2]:
# ─── FIGURA 1: Série temporal duplo eixo ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Painel esquerdo: série temporal
ax1 = axes[0]
ax2 = ax1.twinx()

l1, = ax1.plot(df['ano'], df['populacao_rua'] / 1000, 'o-',
               color='#2980b9', linewidth=2.5, markersize=9,
               label='Pop. sem-teto EUA (mil pessoas)')
l2, = ax2.plot(df['ano'], df['fortuna_bilhoes_usd'], 's--',
               color='#e74c3c', linewidth=2.5, markersize=9,
               label='Fortuna de Musk (bi USD)')

ax1.set_xlabel('Ano', fontsize=11)
ax1.set_ylabel('Pop. sem-teto EUA (mil pessoas)', color='#2980b9', fontsize=11)
ax2.set_ylabel('Fortuna de Musk (bilhões USD)', color='#e74c3c', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#2980b9')
ax2.tick_params(axis='y', labelcolor='#e74c3c')

# Anotações de fases
ax1.axvspan(2010, 2016, alpha=0.06, color='#27ae60')
ax1.text(2012.5, df['populacao_rua'].max() / 1000 * 0.97,
         'Redução\n(Housing First)', ha='center', fontsize=8.5,
         color='#27ae60', style='italic')
ax1.axvspan(2019, 2024, alpha=0.06, color='#e74c3c')
ax1.text(2021.5, df['populacao_rua'].max() / 1000 * 0.97,
         'Aceleração\npós-COVID', ha='center', fontsize=8.5,
         color='#c0392b', style='italic')

lines = [l1, l2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=9)
ax1.set_title('Evolução da Fortuna de Musk\ne da Pop. Sem-Teto nos EUA', fontsize=12, fontweight='bold')

# Painel direito: scatter com regressão
ax = axes[1]
scatter = ax.scatter(df['fortuna_bilhoes_usd'], df['populacao_rua'] / 1000,
                     c=df['ano'], cmap='RdYlGn_r', s=130, zorder=5, edgecolors='white', linewidths=1)
plt.colorbar(scatter, ax=ax, label='Ano', shrink=0.8)

for _, row in df.iterrows():
    ax.annotate(str(int(row['ano'])),
                (row['fortuna_bilhoes_usd'], row['populacao_rua'] / 1000),
                textcoords='offset points', xytext=(6, 4), fontsize=8.5, color='#2c3e50')

# Linha de tendência
z = np.polyfit(df['fortuna_bilhoes_usd'], df['populacao_rua'] / 1000, 1)
p = np.poly1d(z)
x_line = np.linspace(df['fortuna_bilhoes_usd'].min(), df['fortuna_bilhoes_usd'].max(), 200)
ax.plot(x_line, p(x_line), '--', color='#7f8c8d', linewidth=2, alpha=0.8,
        label=f'Tendência linear (r={r_pearson:.2f})')

ax.set_xlabel('Fortuna de Musk (bilhões USD)', fontsize=11)
ax.set_ylabel('Pop. sem-teto nos EUA (mil pessoas)', fontsize=11)
ax.set_title(f'Correlação: Riqueza de Musk × Sem-Teto nos EUA\n'
             f'Pearson r = {r_pearson:.3f} | Spearman ρ = {r_spearman:.3f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

axes[1].text(0.98, -0.1,
             'Fonte: HUD – Annual Homelessness Assessment Reports (2010–2024) | Forbes/Bloomberg',
             transform=axes[1].transAxes, fontsize=8, ha='right', color='gray')

plt.suptitle('Correlação: Crescimento da Fortuna de Musk × Crescimento da\nPopulação em Situação de Rua nos Estados Unidos (2010–2024)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA + 'output_14_correlacao_rua_eua.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: data/output_14_correlacao_rua_eua.png")


Figura salva: data/output_14_correlacao_rua_eua.png


In [3]:
# ─── FIGURA 2: Comparação Brasil vs EUA ────────────────────────────
rua_br = pd.read_csv(DATA + 'historico_pop_rua_brasil.csv')
df_br  = pd.merge(rua_br[['ano', 'populacao_rua']], musk_ano, on='ano', how='inner')
df_br  = df_br.sort_values('ano').reset_index(drop=True)

r_brasil  = np.corrcoef(df_br['fortuna_bilhoes_usd'], df_br['populacao_rua'])[0, 1]
r_eua_val = r_pearson

# Normalizar para comparação (índice base 100 no primeiro ano disponível)
df_eua_plot = df.copy()
df_eua_plot['rua_idx']  = df_eua_plot['populacao_rua']  / df_eua_plot['populacao_rua'].iloc[0]  * 100
df_eua_plot['musk_idx'] = df_eua_plot['fortuna_bilhoes_usd'] / df_eua_plot['fortuna_bilhoes_usd'].iloc[0] * 100

df_br_plot = df_br.copy()
df_br_plot['rua_idx']  = df_br_plot['populacao_rua']  / df_br_plot['populacao_rua'].iloc[0]  * 100
df_br_plot['musk_idx'] = df_br_plot['fortuna_bilhoes_usd'] / df_br_plot['fortuna_bilhoes_usd'].iloc[0] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Painel 1: EUA (índice base 100)
ax = axes[0]
ax.plot(df_eua_plot['ano'], df_eua_plot['musk_idx'],  'o-', color='#e74c3c',
        linewidth=2.5, markersize=8, label='Fortuna de Musk (base 100)')
ax.plot(df_eua_plot['ano'], df_eua_plot['rua_idx'],  's-', color='#2980b9',
        linewidth=2.5, markersize=8, label='Pop. sem-teto EUA (base 100)')
ax.axhline(100, color='gray', linestyle=':', alpha=0.5)
ax.set_yscale('log')
ax.set_ylabel('Índice (base 100 = primeiro ano)', fontsize=11)
ax.set_title('EUA: Variação Relativa\nFortuna de Musk vs. Sem-Teto', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.text(0.03, 0.97,
        f'Pearson r = {r_eua_val:.3f}\nSpearman ρ = {r_spearman:.3f}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='#eaf4fb', alpha=0.8))

# Painel 2: Comparação correlações
ax2 = axes[1]
countries  = ['Brasil\n(2012–2025)', 'EUA\n(2010–2024)']
correlacoes = [r_brasil, r_eua_val]
cores       = ['#27ae60' if v > 0.8 else '#e67e22' if v > 0.5 else '#e74c3c'
               for v in correlacoes]
bars = ax2.bar(countries, correlacoes, color=cores, width=0.45, edgecolor='white')
ax2.set_ylim(-0.1, 1.1)
ax2.set_ylabel('Coeficiente de Pearson (r)', fontsize=11)
ax2.set_title('Comparação: Correlação Musk × Sem-Teto\nBrasil vs. EUA', fontsize=12, fontweight='bold')
ax2.axhline(0.7,  color='#27ae60', linestyle='--', alpha=0.6, linewidth=1.5, label='Forte (0.70)')
ax2.axhline(0.4,  color='#e67e22', linestyle='--', alpha=0.6, linewidth=1.5, label='Moderada (0.40)')
ax2.legend(fontsize=9)
for bar, val in zip(bars, correlacoes):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
             f'r = {val:.3f}', ha='center', fontsize=13, fontweight='bold')

interpretacoes = ['Muito forte\n(ambos crescem juntos)',
                  'Moderada\n(EUA reduziu 2010-16,\nacelerou pós-2019)']
for bar, txt in zip(bars, interpretacoes):
    ax2.text(bar.get_x() + bar.get_width() / 2, 0.05, txt,
             ha='center', va='bottom', fontsize=8, color='#555', style='italic')

axes[1].text(0.98, -0.1,
             'Fonte: IPEA | OBPopRua/UFMG | HUD AHAR | Forbes/Bloomberg',
             transform=axes[1].transAxes, fontsize=8, ha='right', color='gray')

plt.suptitle('Brasil vs. EUA: Como a Riqueza de Musk se Relaciona com o Crescimento do Sem-Teto?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA + 'output_14b_brasil_eua_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: data/output_14b_brasil_eua_comparacao.png")


Figura salva: data/output_14b_brasil_eua_comparacao.png


In [4]:
print("=" * 70)
print("ANÁLISE: CORRELAÇÃO FORTUNA MUSK × POP. SEM-TETO NOS EUA")
print("=" * 70)

anos_disponiveis = list(df['ano'].astype(int))
print(f"\nAnos analisados: {anos_disponiveis}")
print(f"\nEvolução da pop. sem-teto nos EUA (HUD Point-in-Time Count):")
print(f"{'Ano':>6} {'Sem-teto':>10} {'Musk (bi USD)':>15} {'Var. anual':>12}")
print("-" * 50)
for i, row in df.iterrows():
    var = ""
    if i > 0:
        delta = (row['populacao_rua'] - df.iloc[i-1]['populacao_rua'])
        var = f"{'+' if delta > 0 else ''}{delta:,.0f}"
    print(f"{int(row['ano']):>6} {int(row['populacao_rua']):>10,} {row['fortuna_bilhoes_usd']:>15,.0f} {var:>12}")

print(f"\nCorrelação de Pearson:  {r_pearson:.4f}")
print(f"Correlação de Spearman: {r_spearman:.4f}")

if r_pearson > 0.8:
    interp = "positiva MUITO FORTE"
elif r_pearson > 0.6:
    interp = "positiva FORTE"
elif r_pearson > 0.4:
    interp = "positiva MODERADA"
elif r_pearson > 0.2:
    interp = "positiva FRACA"
else:
    interp = "muito fraca ou nula"

print(f"\nInterpretação Pearson: correlação {interp}")

# Fases — usa somente anos realmente disponíveis no dataframe após o merge
def get_row(year):
    row = df[df['ano'] == year]
    return row.iloc[0] if len(row) > 0 else None

ano_ini  = df.iloc[0]
ano_fim  = df.iloc[-1]
ano_2016 = get_row(2016)
ano_2019 = get_row(2019)

print("\n— FASES DA EVOLUÇÃO —")
if ano_2016 is not None:
    queda = (ano_2016['populacao_rua'] / ano_ini['populacao_rua'] - 1) * 100
    print(f"{int(ano_ini['ano'])}–2016: sem-teto {queda:+.1f}%"
          f"  ({int(ano_ini['populacao_rua']):,} → {int(ano_2016['populacao_rua']):,})")
    print(f"         Musk: US$ {ano_ini['fortuna_bilhoes_usd']:.0f}bi "
          f"→ US$ {ano_2016['fortuna_bilhoes_usd']:.0f}bi")

if ano_2019 is not None:
    cresc_rua  = (ano_fim['populacao_rua']          / ano_2019['populacao_rua']          - 1) * 100
    cresc_musk = (ano_fim['fortuna_bilhoes_usd'] / ano_2019['fortuna_bilhoes_usd'] - 1) * 100
    print(f"\n2019–{int(ano_fim['ano'])}: sem-teto {cresc_rua:+.1f}%"
          f"  ({int(ano_2019['populacao_rua']):,} → {int(ano_fim['populacao_rua']):,})")
    print(f"         Musk: US$ {ano_2019['fortuna_bilhoes_usd']:.0f}bi "
          f"→ US$ {ano_fim['fortuna_bilhoes_usd']:.0f}bi  ({cresc_musk:+.0f}%)")

print(f"\n⚠️  A correlação nos EUA é mais complexa que no Brasil (r≈0.97):")
print(f"   › {int(ano_ini['ano'])}–2016: políticas de Housing First reduziram o sem-teto")
print(f"     mesmo com crescimento da riqueza de Musk → correlação NEGATIVA nesse período")
print(f"   › 2019–2024: colapso do mercado imobiliário, inflação de aluguéis,")
print(f"     fim de proteções COVID → correlação POSITIVA forte")
print(f"   › O contraste mostra que políticas públicas IMPORTAM: quando há")
print(f"     vontade política, é possível reduzir a desigualdade de moradia")
print(f"     mesmo num contexto de crescente concentração de riqueza.")

print("\nFonte: HUD AHAR 2010–2024 | Forbes | Bloomberg Billionaires Index")


ANÁLISE: CORRELAÇÃO FORTUNA MUSK × POP. SEM-TETO NOS EUA

Anos analisados: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2022, 2023, 2024]

Evolução da pop. sem-teto nos EUA (HUD Point-in-Time Count):
   Ano   Sem-teto   Musk (bi USD)   Var. anual
--------------------------------------------------
  2012    633,782               2             
  2013    610,042               4      -23,740
  2014    578,424               9      -31,618
  2015    564,708              13      -13,716
  2016    549,928              11      -14,780
  2017    553,742              14       +3,814
  2018    552,830              23         -912
  2019    567,715              20      +14,885
  2020    580,466             165      +12,751
  2022    582,462             137       +1,996
  2023    653,104             232      +70,642
  2024    771,480             432     +118,376

Correlação de Pearson:  0.8401
Correlação de Spearman: 0.2727

Interpretação Pearson: correlação positiva MUITO FORTE

— FASES 